# Court Timeliness Monitor — explorationSynthetic data. Nothing here describes a real court.This notebook is the *exploration* — the looking-around that happened before thepipeline in `src/` was written. The pipeline is the reproducible version; this isthe working out.

In [ ]:
import syssys.path.insert(0, "../src")import pandas as pdimport matplotlib.pyplot as pltfrom config import CASES_RAW, COURTS_RAW, DATE_FORMAT, NA_VALUESpd.set_option("display.width", 120)

## 1. Load, without assuming anythingIDs as text — `"007"` read as a number becomes `7` and the join fails.Dates day-first and explicit — the default parse corrupts only the 1st to 12th ofeach month, which is far harder to spot than everything being wrong.

In [ ]:
cases = pd.read_csv(CASES_RAW, dtype={"case_id": "string", "court_id": "string"},                    na_values=NA_VALUES)for c in ("filed_date", "disposed_date"):    cases[c] = pd.to_datetime(cases[c], format=DATE_FORMAT, errors="coerce")courts = pd.read_csv(COURTS_RAW, dtype={"court_id": "string"})print(cases.shape, courts.shape)cases.head()

## 2. Profile before touchingWhat is one row supposed to be? One case. Is it?

In [ ]:
print("rows          ", len(cases))print("unique case_id", cases["case_id"].nunique())print("duplicated    ", cases.duplicated(subset=["case_id"]).sum())cases.info()

In [ ]:
(cases.isna().mean() * 100).round(1).sort_values(ascending=False)

908 blank disposal dates. That is **not** missing data — those cases are open.Filling them would invent a disposal that never happened.

In [ ]:
cases["case_type"].value_counts(dropna=False)

21 spellings of four categories. Casing, whitespace, and abbreviations.

In [ ]:
cases[cases.duplicated(subset=["case_id"], keep=False)].sort_values("case_id").head(6)

Some duplicate pairs disagree — one row open, one disposed. That looks like asuperseded extract, so the more complete record is the one to keep. It is adecision, and it gets logged as one.

## 3. The reconciliationThe extract supplies `reported_days`. The duration can also be derived from thetwo dates. Do they agree?

In [ ]:
reported = pd.to_numeric(cases["reported_days"].astype("string").str.strip(),                         errors="coerce")calculated = (cases["disposed_date"] - cases["filed_date"]).dt.daysboth = reported.notna() & calculated.notna()mismatch = both & (reported != calculated)print(f"{mismatch.sum()} cases where the two disagree")cases.loc[mismatch, ["case_id", "filed_date", "disposed_date", "reported_days"]].head()

47 disagreements. Only findable by checking one source against an independentone — the same control an accountant applies when a ledger has to tie to astatement.These are flagged, **not corrected**. Silently overwriting them would hide aproblem in the source system rather than surface it.

## 4. Impossible values

In [ ]:
print("disposed before filed:", (calculated < 0).sum())print("filed in the future:  ", (cases["filed_date"] > pd.Timestamp("2026-09-01")).sum())print("court_id not in lookup:", (~cases["court_id"].isin(courts["court_id"])).sum())

## 5. The merge — assert the assumption`validate="m:1"` raises immediately if the lookup key is not unique, so afan-out fails loudly instead of quietly tripling every total.

In [ ]:
before = len(cases)merged = cases.merge(courts, on="court_id", how="left", validate="m:1", indicator=True)print(before, "->", len(merged))merged["_merge"].value_counts()

## 6. Distribution — before choosing a headline number

In [ ]:
w = (merged["disposed_date"] - merged["filed_date"]).dt.daysw = w[w > 0]fig, ax = plt.subplots(figsize=(9, 3.6))ax.hist(w, bins=45, color="#2a78d6")ax.axvline(w.median(), color="black", lw=1.5)ax.axvline(w.mean(), color="#eb6834", lw=1.5, ls="--")ax.set_xlabel("Days to disposal")print(f"median {w.median():.0f}   mean {w.mean():.0f}")

Long right tail. The mean sits above the median, describing a case that mostlydoes not exist. Every headline figure in this project is a median.

## 7. The thing that did not make senseMedian by filing quarter. Watch the most recent periods.

In [ ]:
tmp = merged.assign(q=merged["filed_date"].dt.to_period("Q").astype(str),                    days=(merged["disposed_date"] - merged["filed_date"]).dt.days)e = tmp[tmp["region"] == "Eastern"]by_q = e.groupby("q").agg(filed=("case_id", "nunique"),                          closed=("disposed_date", "count"),                          median_days=("days", "median"))by_q["closure_rate"] = (by_q["closed"] / by_q["filed"]).round(2)by_q

The median falls sharply in the last quarters — which looks like a bigimprovement.It is not. Look at `closure_rate`: 94% of the earliest quarter's filings haveclosed, against 21% of the most recent. **Only closed cases can be measured**, andin recent quarters only the fast ones have closed. The slow cases are still openand invisible to the measure.This is right-censoring, and it is the single most important thing in the dataset.A dashboard reporting that series without a completeness check would show animprovement that has not happened.**What the pipeline does about it:** computes the closure rate per quarter, flagsanything under 80% as incomplete, and reports the open backlog's age profilealongside — the measure that *does* see the slow cases.From here the logic moved into `src/`, where it can be re-run.